# Step 1 - Environment setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB" if torch.cuda.is_available() else "")

CUDA available: True
GPU name: Tesla T4
VRAM: 15.64 GB


In [2]:
!pip install -q transformers datasets accelerate huggingface_hub

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from huggingface_hub import login

print("torch version      :", torch.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("transformers OK    : yes")
print("datasets OK        : yes")
print("huggingface_hub OK : yes")

torch version      : 2.10.0+cu128
CUDA available     : True
transformers OK    : yes
datasets OK        : yes
huggingface_hub OK : yes


In [ ]:
login(token="hugging_face_access_token")

# Step 2 - Load & explore data

In [5]:
from datasets import load_dataset

dataset = load_dataset("xlangai/spider")

print(dataset)

README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [6]:
for i in range(3):
    example = dataset["train"][i]
    print(f"--- Example {i+1} ---")
    print("Question :", example["question"])
    print("SQL      :", example["query"])
    print("DB       :", example["db_id"])
    print()

--- Example 1 ---
Question : How many heads of the departments are older than 56 ?
SQL      : SELECT count(*) FROM head WHERE age  >  56
DB       : department_management

--- Example 2 ---
Question : List the name, born state and age of the heads of departments ordered by age.
SQL      : SELECT name ,  born_state ,  age FROM head ORDER BY age
DB       : department_management

--- Example 3 ---
Question : List the creation year, name and budget of each department.
SQL      : SELECT creation ,  name ,  budget_in_billions FROM department
DB       : department_management



In [7]:
print("Column names:", dataset["train"].column_names)
print()
print("Train size     :", len(dataset["train"]))
print("Validation size:", len(dataset["validation"]))

Column names: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks']

Train size     : 7000
Validation size: 1034


In [8]:
lengths = [len(ex["query"].split()) for ex in dataset["train"]]

print("Shortest SQL (words) :", min(lengths))
print("Longest SQL  (words) :", max(lengths))
print("Average SQL  (words) :", round(sum(lengths) / len(lengths), 1))

short  = sum(1 for l in lengths if l <= 10)
medium = sum(1 for l in lengths if 10 < l <= 30)
long_  = sum(1 for l in lengths if l > 30)

print()
print(f"Short  (<=10 words) : {short}  ({round(short/len(lengths)*100)}%)")
print(f"Medium (11-30 words): {medium} ({round(medium/len(lengths)*100)}%)")
print(f"Long   (>30 words)  : {long_}  ({round(long_/len(lengths)*100)}%)")

Shortest SQL (words) : 4
Longest SQL  (words) : 87
Average SQL  (words) : 15.9

Short  (<=10 words) : 2522  (36%)
Medium (11-30 words): 4020 (57%)
Long   (>30 words)  : 458  (7%)


In [9]:
def format_example(example):
    return f"Question: {example['question']}\nSQL: {example['query']}"

for i in range(3):
    print(f"--- Formatted Example {i+1} ---")
    print(format_example(dataset["train"][i]))
    print()

--- Formatted Example 1 ---
Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56

--- Formatted Example 2 ---
Question: List the name, born state and age of the heads of departments ordered by age.
SQL: SELECT name ,  born_state ,  age FROM head ORDER BY age

--- Formatted Example 3 ---
Question: List the creation year, name and budget of each department.
SQL: SELECT creation ,  name ,  budget_in_billions FROM department



# Step 3 - Preprocess & format

In [10]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

print("Vocab size :", tokenizer.vocab_size)
print("Pad token  :", tokenizer.pad_token)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab size : 50257
Pad token  : <|endoftext|>


In [11]:
MAX_LENGTH = 128

def preprocess(example):
    text = f"Question: {example['question']}\nSQL: {example['query']}{tokenizer.eos_token}"
    tokenized = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1034
    })
})


In [12]:
sample = tokenized_dataset["train"][0]

print("Keys            :", list(sample.keys()))
print("input_ids length:", len(sample["input_ids"]))
print("labels length   :", len(sample["labels"]))
print()

decoded = tokenizer.decode(sample["input_ids"], skip_special_tokens=False)
print("Decoded sample:\n", decoded[:300])

Keys            : ['input_ids', 'attention_mask', 'labels']
input_ids length: 128
labels length   : 128

Decoded sample:
 Question: How many heads of the departments are older than 56 ?
SQL: SELECT count(*) FROM head WHERE age  >  56<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endof


In [13]:
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Train sample types:")
sample = tokenized_dataset["train"][0]
for k, v in sample.items():
    print(f"  {k}: {v.shape}, dtype={v.dtype}")

Train sample types:
  input_ids: torch.Size([128]), dtype=torch.int64
  attention_mask: torch.Size([128]), dtype=torch.int64
  labels: torch.Size([128]), dtype=torch.int64


# Step 4 - Configure training

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sql-gpt2-medium",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
)

print("Training config ready")
print(f"  Epochs          : {training_args.num_train_epochs}")
print(f"  Batch size      : {training_args.per_device_train_batch_size}")
print(f"  Warmup steps    : {training_args.warmup_steps}")
print(f"  FP16            : {training_args.fp16}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training config ready
  Epochs          : 3
  Batch size      : 8
  Warmup steps    : 100
  FP16            : True


In [15]:
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer

model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("Model parameters:", round(model.num_parameters() / 1e6, 1), "M")
print("Trainer ready")

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model parameters: 354.8 M
Trainer ready


# Step 5 - Full fine-tune

In [16]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.863368,1.443231
2,0.674488,1.613382
3,0.579680,1.707548


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=2625, training_loss=0.8176555037725539, metrics={'train_runtime': 1448.6919, 'train_samples_per_second': 14.496, 'train_steps_per_second': 1.812, 'total_flos': 4875678646272000.0, 'train_loss': 0.8176555037725539, 'epoch': 3.0})

## Rerun with updated arguments for training

In [18]:
training_args = TrainingArguments(
    output_dir="./sql-gpt2-medium",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=100,
    weight_decay=0.05,        # increased from 0.01
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    learning_rate=1e-5,       # added — default 5e-5 is too aggressive
    max_grad_norm=0.5,        # added — clips large gradient updates
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [19]:
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, Trainer

model = GPT2LMHeadModel.from_pretrained("gpt2-medium")
model.resize_token_embeddings(len(tokenizer))

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
)

print("Model parameters:", round(model.num_parameters() / 1e6, 1), "M")
print("Trainer ready")

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model parameters: 354.8 M
Trainer ready


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.134478,1.410340
2,1.006180,1.417961
3,0.968459,1.434412


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=2625, training_loss=1.1494669145856584, metrics={'train_runtime': 1513.4251, 'train_samples_per_second': 13.876, 'train_steps_per_second': 1.734, 'total_flos': 4875678646272000.0, 'train_loss': 1.1494669145856584, 'epoch': 3.0})

# Step 6 - Evaluate & test

In [27]:
import os

checkpoints = [d for d in os.listdir("./sql-gpt2-medium") if d.startswith("checkpoint")]
checkpoints.sort()
for c in checkpoints:
    print(c)

checkpoint-1750
checkpoint-2625
checkpoint-875


In [28]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

best_checkpoint = "./sql-gpt2-medium/checkpoint-875"  # replace with actual name

model = GPT2LMHeadModel.from_pretrained(best_checkpoint)
tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

model.eval()
print("Loaded checkpoint:", best_checkpoint)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loaded checkpoint: ./sql-gpt2-medium/checkpoint-875


In [29]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def generate_sql(question, max_new_tokens=64):
    prompt = f"Question: {question}\nSQL:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            do_sample=False,        # greedy decoding — deterministic output
            repetition_penalty=1.2  # penalizes repeating tokens
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    sql_part = decoded.split("SQL:")[-1].strip()
    return sql_part

# Test on a few questions
test_questions = [
    "How many singers are there?",
    "What is the average age of all singers?",
    "List all singer names ordered by age.",
]

for q in test_questions:
    print(f"Question : {q}")
    print(f"Generated: {generate_sql(q)}")
    print()

Question : How many singers are there?
Generated: SELECT count(*) FROM singer GROUP BY artist ORDER by COUNT (*) DESC LIMIT 1; Query : What is the average number of songs for each genre and how long does it take to play them all. SQL  = "SELECT avg_songtime , AVG (duration)" WHERE genres LIKE 'SING

Question : What is the average age of all singers?
Generated: :SELECT T1.name ,  COUNT(*) AS t2, sum(T1._idx ) as ct from song WHERE name LIKE "%s%"

Question : List all singer names ordered by age.
Generated: SELECT Name FROM song ORDER BY Age DESC LIMIT 1; Query : Show All Songs  ASC , NOT EXCEPT AS INTERSECT GROUP By Song_Name HAVING COUNT(*) BETWEEN 0 AND (SELECT max((Age) - MINIMUM1))%MINUS 100 WHERE Artist !=



### Evaluation 1

In [25]:
correct = 0
total = 50

for i in range(total):
    example = dataset["validation"][i]
    generated = generate_sql(example["question"]).lower().strip()
    gold = example["query"].lower().strip()
    if generated == gold:
        correct += 1

print(f"Exact match accuracy: {correct}/{total} = {round(correct/total*100, 1)}%")

Exact match accuracy: 0/50 = 0.0%


In [26]:
print("Sample predictions vs ground truth\n")
for i in range(5):
    example = dataset["validation"][i]
    generated = generate_sql(example["question"])
    print(f"Question : {example['question']}")
    print(f"Gold SQL : {example['query']}")
    print(f"Generated: {generated}")
    print()

Sample predictions vs ground truth

Question : How many singers do we have?
Gold SQL : SELECT count(*) FROM singer
Generated: SELECT count(*) FROM singer GROUP BY artist ORDER by COUNT (*) DESC LIMIT 1; Query : What are the names of all songs that contain "I'm in Love" or any other song with a title ending in ".love"? SQL ------------ EXCEPTION code ----------- 'SELECT T1.

Question : What is the total number of singers?
Gold SQL : SELECT count(*) FROM singer
Generated: SELECT count(*) FROM singer GROUP BY actor ORDER by COUNT (*) DESC LIMIT 1; Query result for each song in descending order. SQLite3::SELECT T1 .song_id ,  sum(T2 ) AS tl from artist WHERE name LIKE '%s' INTERSECT EX

Question : Show name, country, age for all singers ordered by age from the oldest to the youngest.
Gold SQL : SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Generated: SELECT T1 .name ,  COUNT(*) FROM singer AS t2 JOIN song ORDER BY Age ASC LIMIT 1; EXCEPT SQL_Exception "SELECT Name , Country ,

### Evaluation 2

In [30]:
import re

def normalize_sql(sql):
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)        # collapse multiple spaces
    sql = sql.rstrip(';')                  # remove trailing semicolon
    return sql

correct = 0
total = 50

for i in range(total):
    example = dataset["validation"][i]
    generated = normalize_sql(generate_sql(example["question"]))
    gold = normalize_sql(example["query"])
    if generated == gold:
        correct += 1

print(f"Normalized exact match: {correct}/{total} = {round(correct/total*100, 1)}%")

Normalized exact match: 0/50 = 0.0%


In [31]:
print("Sample predictions vs ground truth\n")
for i in range(5):
    example = dataset["validation"][i]
    generated = generate_sql(example["question"])
    print(f"Question : {example['question']}")
    print(f"Gold SQL : {example['query']}")
    print(f"Generated: {generated}")
    print()

Sample predictions vs ground truth

Question : How many singers do we have?
Gold SQL : SELECT count(*) FROM singer
Generated: SELECT count(*) FROM singer GROUP BY artist ORDER by COUNT (*) DESC LIMIT 1; Query : What are the names of all songs that contain "I'm in Love" or any other song with a title ending in ".love"? SQL ------------ EXCEPTION code ----------- 'SELECT T1.

Question : What is the total number of singers?
Gold SQL : SELECT count(*) FROM singer
Generated: SELECT count(*) FROM singer GROUP BY actor ORDER by COUNT (*) DESC LIMIT 1; Query result for each song in descending order. SQLite3::SELECT T1 .song_id ,  sum(T2 ) AS tl from artist WHERE name LIKE '%s' INTERSECT EX

Question : Show name, country, age for all singers ordered by age from the oldest to the youngest.
Gold SQL : SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Generated: SELECT T1 .name ,  COUNT(*) FROM singer AS t2 JOIN song ORDER BY Age ASC LIMIT 1; EXCEPT SQL_Exception "SELECT Name , Country ,

# Step 7 - Push to HF Hub

In [32]:
from huggingface_hub import HfApi

model_name = "gpt2-medium-sql-generator"  # will appear as your-username/gpt2-medium-sql-generator

model.push_to_hub(model_name)
tokenizer.push_to_hub(model_name)

print(f"Model pushed to: https://huggingface.co/poseidon1113/{model_name}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...eabnnt1/model.safetensors:   0%|          |  550kB / 1.42GB            

README.md: 0.00B [00:00, ?B/s]

Model pushed to: https://huggingface.co/poseidon1113/gpt2-medium-sql-generator


In [34]:
from huggingface_hub import HfApi

api = HfApi()

model_card = """
---
language: en
tags:
- text-to-sql
- gpt2
- fine-tuned
- sql-generation
datasets:
- xlangai/spider
---

# GPT-2 Medium — SQL Query Generator

Fine-tuned GPT-2 Medium on the Spider text-to-SQL dataset to generate SQL queries from natural language questions.

## Training
- Base model: GPT-2 Medium (354M parameters)
- Dataset: Spider (7000 train / 1034 validation examples)
- Method: Full fine-tuning
- Best checkpoint: Epoch 1 (val loss 1.410)

## Usage
```python
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained("your-username/gpt2-medium-sql-generator")
tokenizer = GPT2Tokenizer.from_pretrained("your-username/gpt2-medium-sql-generator")

prompt = "Question: How many singers are there?\\nSQL:"
inputs = tokenizer(prompt, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))
```

## Limitations
- GPT-2 is a small model — output SQL may hallucinate table/column names
- No schema awareness — works best on Singer/Concert domain from Spider training data
- Intended as a learning project demonstrating full fine-tuning pipeline
"""

api.upload_file(
    path_or_fileobj=model_card.encode(),
    path_in_repo="README.md",
    repo_id=f"poseidon1113/{model_name}",
    repo_type="model"
)

print("Model card uploaded")

Model card uploaded
